<a href="https://colab.research.google.com/github/Abdulkadir03/Crop-Recommender-for-Sahel-Region/blob/main/notebooks/09b_field_validation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 09b: Field validation of the models

**Purpose:** Test the out-of-fold predictions of the random forest baseline (notebook 08) and the LightGBM per-crop models (notebook 09) against independent field observations (CropHarvest points; Niger and Nigeria household surveys), and compare them with naive baselines. This is the check against real fields.

**Run after:** notebooks 08 and 09 (this notebook reads their saved out-of-fold predictions).

**Inputs:** `oof_rf_env.parquet` (08), `oof_lgb_env.parquet` (09), `training_table.parquet` and `training_config.json` (07), `ground_truth_cropharvest_pixels.parquet` (05b), `ground_truth_surveys_sites.parquet` and `ground_truth_surveys_crops.parquet` (05c).

**Method:**
1. Predictions are out-of-fold: each pixel is scored by a model that did not see its spatial fold.
2. Surveys (Niger, Nigeria): observed share of plot area per crop, averaged per SPAM pixel. Per crop: rank correlation between predicted and observed share, and AUC for presence versus absence where both classes have at least 10 pixels, with bootstrap 95% intervals over pixels.
3. CropHarvest: presence-only evidence. For each (pixel, crop) pair, is the observed crop among the model's top 5 of 16 crops (chance 31%), and what is its mean rank?
4. Comparison with two naive baselines built from the training folds: global mean share and mean share per country. The country baseline is nearly constant within a country, so its AUC within a country is about 0.5 by construction.
5. A relative-suitability score (predicted share divided by the crop's training-fold mean share) is evaluated alongside the raw predicted share.

**Outputs:** none saved; results are summarised in the README.

**Results**

*Household surveys: presence AUC (random forest / LightGBM), crops with at least 10 pixels in each class*

| Country | Crop | RF | LightGBM |
|---|---|---|---|
| Niger (297 pixels) | sorghum | 0.80 | 0.78 |
| | cowpea | 0.78 | 0.84 |
| | groundnut | 0.64 | 0.68 |
| | sesame | 0.68 | 0.68 |
| | cassava | 0.74 | 0.80 |
| | rice | 0.61 | 0.63 |
| | maize | 0.64 | 0.66 |
| | onion | 0.61 | 0.57 |
| | tomato | 0.74 | 0.69 |
| Nigeria (135 pixels) | pearl millet | 0.86 | 0.84 |
| | soybean | 0.77 | 0.71 |
| | maize | 0.69 | 0.64 |
| | cowpea | 0.69 | 0.62 |
| | rice | 0.65 | 0.62 |
| | groundnut | 0.64 | 0.67 |
| | cassava | 0.56 | 0.49 |
| | onion | 0.56 | 0.46 |
| | sesame | 0.48 | 0.47 |

- The bootstrap intervals of the two models overlap for every crop, so none of the differences is statistically meaningful. LightGBM is higher for 5 of 9 Niger crops, the random forest for 8 of 9 Nigerian crops.
- Skill demonstrated by both models (interval above 0.5): Niger sorghum, cowpea, groundnut, sesame, cassava and (borderline) rice and tomato; Nigeria pearl millet, soybean, maize, groundnut and rice. No demonstrated skill: Niger maize and onion; Nigeria cassava, sesame and onion (and cowpea for LightGBM). Cotton, sugarcane, sweet potato and yams have too few pixels.

*CropHarvest (Burkina Faso, Mali, Senegal): top-5 hit rate of the observed crop*

| Group (pixel-crop pairs) | RF | LightGBM | Country mean | Global mean | LightGBM relative score |
|---|---|---|---|---|---|
| All crops (353) | 0.765 | 0.765 | 0.799 | 0.754 | 0.663 |
| Excluding the five most common (87) | 0.218 | 0.195 | 0.345 | 0.000 | 0.437 |
| The five most common (266) | 0.944 | 0.951 | 0.947 | 1.000 | 0.737 |

- Neither model beats the country mean in these three countries, where country-level patterns are already built into SPAM and the field data.
- The relative score lifts the hit rate for less common crops (0.20 to 0.44, above the country mean of 0.35) but lowers it for the five most common (0.95 to 0.74), mainly sorghum (1.00 to 0.51). It also helps cotton and sesame.
- Rice is poorly placed by every model (hit rate 0.06 to 0.17 against 0.57 for the country mean), and the relative score does not help.

**Notes:**
- Sample sizes are small (counted in pixels): Niger 297, Nigeria 135, CropHarvest 93. Differences of a few points are not significant.
- Survey locations are anonymised or offset, and the observation years (2017 to 2022) differ from SPAM's 2019 to 2021.
- Neither the raw nor the relative score is yet a finished recommendation rule; the choice is revisited after the agronomic constraints are added.

In [ ]:
# @title Cell 1: predictions and baselines per pixel
import os, json
import numpy as np, pandas as pd
from scipy.stats import spearmanr
from sklearn.metrics import roc_auc_score
from google.colab import drive
drive.mount('/content/drive')

BASE = '/content/drive/MyDrive'
P = f'{BASE}/crop_project'
cfg = json.load(open(f'{P}/training_config.json'))
CROPS = cfg['crops']
SHARE = [f'share_{c}' for c in CROPS]

tt = pd.read_parquet(f'{P}/training_table.parquet', columns=['px', 'py', 'fold', 'adm0', 'has_label'] + SHARE)
pix = (tt[tt.has_label].groupby(['px', 'py'])
       .agg(fold=('fold', 'first'), adm0=('adm0', lambda s: s.mode().iat[0]),
            **{f'share_{c}': (f'share_{c}', 'first') for c in CROPS})
       .reset_index())
oof = pd.read_parquet(f'{P}/oof_rf_env.parquet')
pix = pix.merge(oof.drop(columns='fold'), on=['px', 'py'], how='left')      # adds 16 prediction columns

S = pix[SHARE].to_numpy('float64'); fold = pix.fold.to_numpy(); ctry = pix.adm0.to_numpy()
Pr = pix[CROPS].to_numpy('float64')
Pg, Pc = np.zeros_like(S), np.zeros_like(S)
for k in range(5):
    tr, te = fold != k, fold == k
    gm = S[tr].mean(0); Pg[te] = gm
    means = pd.DataFrame(S[tr]).groupby(ctry[tr]).mean()
    cm = means.reindex(ctry[te])
    Pc[te] = np.where(cm.isna().to_numpy(), gm, cm.to_numpy())

models = {'rf': Pr, 'country': Pc, 'global': Pg}
idx = {f'{a}_{b}': i for i, (a, b) in enumerate(zip(pix.px.astype(int), pix.py.astype(int)))}
if os.path.exists(f'{P}/oof_lgb_env.parquet'):
    o = pd.read_parquet(f'{P}/oof_lgb_env.parquet').drop(columns='fold')
    models['lgb'] = pix[['px', 'py']].merge(o, on=['px', 'py'], how='left')[CROPS].to_numpy('float64')
print('Pixels:', len(pix), '| pixels with missing predictions:', int(np.isnan(Pr).any(axis=1).sum()))

In [ ]:
# @title Cell 2: household surveys (Niger and Nigeria)
sites = pd.read_parquet(f'{P}/ground_truth_surveys_sites.parquet')
scrops = pd.read_parquet(f'{P}/ground_truth_surveys_crops.parquet')
use = sites[sites.within_5km & (sites.has_label == True)].copy()
use['pixel'] = use.px.astype(int).astype(str) + '_' + use.py.astype(int).astype(str)
use = use[use.pixel.isin(idx)]

full = (use[['site_id', 'country', 'pixel']].assign(_k=1)
        .merge(pd.DataFrame({'crop_code': CROPS, '_k': 1}), on='_k').drop(columns='_k')
        .merge(scrops[['site_id', 'crop_code', 'weight_share']], on=['site_id', 'crop_code'], how='left')
        .fillna({'weight_share': 0.0}))
obs = full.groupby(['country', 'pixel', 'crop_code']).weight_share.mean().unstack('crop_code')[CROPS]

def survey_table(country):
    o = obs.xs(country, level='country')
    ii = [idx[p] for p in o.index]
    rows = []
    for j, c in enumerate(CROPS):
        y = o[c].to_numpy(); pres = y > 0
        row = {'crop': c, 'pixels': len(y), 'prevalence': round(float(pres.mean()), 2)}
        for name, M in models.items():
            s = M[ii, j]
            row[f'rho_{name}'] = round(float(spearmanr(y, s).correlation), 3) if y.std() > 0 and s.std() > 0 else np.nan
            ok = pres.sum() >= 10 and (~pres).sum() >= 10 and s.std() > 0
            row[f'auc_{name}'] = round(float(roc_auc_score(pres, s)), 3) if ok else np.nan
        rows.append(row)
    return pd.DataFrame(rows).set_index('crop')

pd.set_option('display.width', 220)
for name in ['Niger', 'Nigeria']:
    print(f'\n{name}: pixels with survey sites =', obs.xs(name, level='country').shape[0])
    print(survey_table(name))

In [ ]:
#@title Relative-suitability scores (predicted share divided by the crop's mean share) { display-mode: "both" }
base = np.maximum(Pg, 1e-6)
models['rf_lift'] = models['rf'] / base
if 'lgb' in models:
    models['lgb_lift'] = models['lgb'] / base
print('Models:', list(models))

In [ ]:
# @title Cell 3: CropHarvest points
ch = pd.read_parquet(f'{P}/ground_truth_cropharvest_pixels.parquet')
ch['pixel'] = ch.px.astype(int).astype(str) + '_' + ch.py.astype(int).astype(str)
ch = ch[ch.pixel.isin(idx) & ch.crop_code.isin(CROPS)].copy()
ci = ch.pixel.map(idx).to_numpy()
cj = ch.crop_code.map({c: j for j, c in enumerate(CROPS)}).to_numpy()
EASY = ['sorg', 'pmil', 'cowp', 'grou', 'maiz']
for name, M in models.items():
    ch[f'rank_{name}'] = (-M).argsort(1).argsort(1)[ci, cj]          # 0 = best of 16

def hits(df, label):
    out = {'group': label, 'pixel_crop_pairs': len(df)}
    for name in models:
        out[f'top5_{name}'] = round(float((df[f'rank_{name}'] < 5).mean()), 3)
        out[f'meanrank_{name}'] = round(float(df[f'rank_{name}'].mean() + 1), 2)
    return out

rows = [hits(ch, 'all crops'),
        hits(ch[~ch.crop_code.isin(EASY)], 'excluding the five most common'),
        hits(ch[ch.crop_code.isin(EASY)], 'the five most common')]
rows += [hits(ch[ch.crop_code == c], c) for c in CROPS if (ch.crop_code == c).sum() >= 10]
print(pd.DataFrame(rows).set_index('group').to_string())
print('\nChance level for a top-5 hit among 16 crops: 0.31 (mean rank 8.5)')
drive.flush_and_unmount(); drive.mount('/content/drive')

In [ ]:
# @title Cell 4: bootstrap intervals
rows = []
for MODEL in [m for m in ['rf', 'lgb'] if m in models]:
    for country in ['Niger', 'Nigeria']:
        o = obs.xs(country, level='country')
        ii = np.array([idx[p] for p in o.index])
        for j, c in enumerate(CROPS):
            y = o[c].to_numpy() > 0
            if y.sum() < 10 or (~y).sum() < 10:
                continue
            s = models[MODEL][ii, j]
            rng = np.random.default_rng(0); vals = []
            for _ in range(500):
                b = rng.integers(0, len(y), len(y))
                if 0 < y[b].sum() < len(y) and s[b].std() > 0:
                    vals.append(roc_auc_score(y[b], s[b]))
            lo, hi = np.percentile(vals, [2.5, 97.5])
            rows.append({'model': MODEL, 'country': country, 'crop': c, 'pixels': len(y),
                         'present': int(y.sum()), 'auc': round(roc_auc_score(y, s), 3),
                         'ci_low': round(lo, 3), 'ci_high': round(hi, 3)})
print(pd.DataFrame(rows).sort_values(['country', 'crop', 'model']).to_string(index=False))